In [ ]:
# 🔬 Inferencia y Modelamiento Predictivo de Anomalías Térmicas
**Pipeline Maestro Unificado para Artículo Científico (Nivel Q1)**

Este script orquesta el benchmarking de modelos, el Estudio de Ablación,
el modelamiento secuencial profundo (LSTM) y la predicción probabilística (TFT).

In [ ]:
# ==============================================================================
# CONFIGURACIÓN DEL ENTORNO, DEPENDENCIAS Y FUENTE ÚNICA DE VERDAD
# ==============================================================================
!pip install -q pytorch-forecasting lightning catboost lightgbm xgboost optuna tabulate

import os
import gc
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 300,
    "font.family": "serif",
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.titlesize": 12
})

# [FUENTE ÚNICA DE VERDAD]: Todas las fases leerán exclusivamente de aquí.
# Queda estrictamente prohibido generar excels o csv intermedios.
DATASET_PATH = "/kaggle/input/datasets/danielchura/huayao-pipeline-dataset/dataset_huayao_preprocessed.csv"

# Limpieza inicial agresiva de la memoria del acelerador gráfico
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("[SISTEMA] Dependencias instaladas y memoria inicializada correctamente.")

In [ ]:
# ==============================================================================
# FASE 1: ANÁLISIS EXPLORATORIO DE DATOS Y CORRELACIONES FÍSICAS
# ==============================================================================
def main_eda():
    print("=" * 75)
    print("  PHASE 1: EXPLORATORY DATA ANALYSIS (EDA) & PHYSICAL CORRELATIONS")
    print("=" * 75)
    
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(f"[ERROR CRÍTICO] No se encontró el dataset en: {DATASET_PATH}")
        
    df_clean = pd.read_csv(DATASET_PATH, parse_dates=["datetime"])
    
    # FIGURA 1: Serie Temporal Histórica
    plt.figure(figsize=(8, 4))
    plt.scatter(df_clean["datetime"], df_clean["TT"], s=1.5, alpha=0.2, color="darkorange")
    plt.title("Serie Temporal Histórica de Temperatura (Huayao)", fontweight="bold")
    plt.xlabel("Línea de Tiempo (2018 - 2025)")
    plt.ylabel("Temperatura Superficial (°C)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("Fig_01_Serie_Historica_TT.png", dpi=300)
    plt.close()

    # FIGURA 2: Distribución de Temperatura
    plt.figure(figsize=(6, 4))
    sns.kdeplot(df_clean["TT"].dropna(), color="crimson", fill=True, alpha=0.3, linewidth=2)
    plt.title("Distribución Estadística de la Temperatura del Aire", fontweight="bold")
    plt.xlabel("Temperatura del Aire (°C)")
    plt.ylabel("Densidad de Probabilidad")
    plt.tight_layout()
    plt.savefig("Fig_02_Distribucion_Imputacion.png", dpi=300)
    plt.close()

    # FIGURA 3: Matriz de Correlación de Spearman
    corr_features = ["TT", "HR", "PP", "RR", "wind_u", "wind_v", "dew_point_dep"]
    plt.figure(figsize=(6, 5))
    corr = df_clean[corr_features].corr(method="spearman")
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap="coolwarm", fmt=".2f", square=True, linewidths=.5, cbar_kws={"shrink": .8})
    plt.title("Matriz de Correlación de Spearman", fontweight="bold")
    plt.tight_layout()
    plt.savefig("Fig_03_Matriz_Correlacion_Spearman.png", dpi=300)
    plt.close()
    
    print("[INFO] Gráficos de auditoría exportados con éxito (Fig 1, 2 y 3).")
    del df_clean
    gc.collect()

main_eda()

In [ ]:
# ==============================================================================
# FASE 2: BASELINES, ESTUDIO DE ABLACIÓN Y SIGNIFICANCIA ESTADÍSTICA
# ==============================================================================
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from scipy.stats import friedmanchisquare, norm, wilcoxon
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

HORIZON = 720  
LAG_HOURS = [1, 24, 72, 168, 360]  
BOOTSTRAP_RESETS = 1000

def generate_optimized_lags(df: pd.DataFrame) -> pd.DataFrame:
    df_lagged = df.copy()
    for lag in LAG_HOURS:
        df_lagged[f"lag_TT_{lag}h"] = df_lagged["TT"].shift(lag)
        df_lagged[f"lag_TT_anomaly_{lag}h"] = df_lagged["TT_anomaly"].shift(lag)
    return df_lagged.dropna()

def run_recursive_inference(model, initial_history, future_exog_df, features, horizon=720):
    predictions = []
    curr_hist = list(initial_history)
    
    lag_cols = [c for c in features if "lag_" in c]
    exo_cols = [c for c in features if "lag_" not in c]
    
    exo_data = future_exog_df[exo_cols].values 
    lag_vals = [int(col.split("_")[-1].replace("h", "")) for col in lag_cols]
    
    x_step = np.zeros((1, len(features)))
    exo_indices = [features.index(col) for col in exo_cols]
    lag_indices = [features.index(col) for col in lag_cols]
    
    for i in range(horizon):
        x_step[0, exo_indices] = exo_data[i]
        for j, lag_val in enumerate(lag_vals):
            x_step[0, lag_indices[j]] = curr_hist[-lag_val]
            
        x_step_df = pd.DataFrame(x_step, columns=features)
        pred = model.predict(x_step_df)[0]
        
        predictions.append(pred)
        curr_hist.append(pred)
        
    return np.array(predictions)

def print_markdown_table(df, title):
    print(f"\n{title}")
    print("=" * len(title))
    try:
        print(df.to_markdown(floatfmt=".4f"))
    except:
        print(df.to_string())

def run_ablation_study(df_lagged):
    print("\n[INFO] Iniciando Estudio de Ablación Física (LightGBM)...")
    
    configs = {
        "A (Base)": {"target": "TT", "features": ["hour_sin", "hour_cos", "month_sin", "month_cos"] + [f"lag_TT_{l}h" for l in LAG_HOURS]},
        "B (+Viento)": {"target": "TT", "features": ["hour_sin", "hour_cos", "month_sin", "month_cos", "wind_u", "wind_v"] + [f"lag_TT_{l}h" for l in LAG_HOURS]},
        "C (+Rocío)": {"target": "TT", "features": ["hour_sin", "hour_cos", "month_sin", "month_cos", "wind_u", "wind_v", "dew_point_dep"] + [f"lag_TT_{l}h" for l in LAG_HOURS]},
        "D (+Anomalía)": {"target": "TT_anomaly", "features": ["hour_sin", "hour_cos", "month_sin", "month_cos", "wind_u", "wind_v", "dew_point_dep"] + [f"lag_TT_anomaly_{l}h" for l in LAG_HOURS]}
    }
    
    df_train = df_lagged[df_lagged["datetime"] < "2024-01-01"]
    start_dates = pd.date_range(start="2025-01-01 00:00:00", end="2025-12-01 00:00:00", freq="MS")
    ablation_maes = {}

    for conf_name, conf in configs.items():
        # Entrenar modelo
        model = LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, verbosity=-1, random_state=42, n_jobs=-1)
        model.fit(df_train[conf["features"]], df_train[conf["target"]])
        
        # Evaluar ventanas 2025
        window_errors = []
        for start_date in start_dates:
            context_data = df_lagged[df_lagged["datetime"] < start_date]
            future_data = df_lagged[df_lagged["datetime"] >= start_date].head(HORIZON)
            y_real = future_data["TT"].values
            
            hist_target = context_data[conf["target"]].tail(360).tolist()
            preds = run_recursive_inference(model, hist_target, future_data, conf["features"], horizon=HORIZON)
            
            # Reconstruir si el target fue la anomalía
            if conf["target"] == "TT_anomaly":
                preds = preds + future_data["TT_climatology"].values
                
            window_errors.append(mean_absolute_error(y_real, preds))
            
        ablation_maes[conf_name] = np.mean(window_errors)
        del model

    # Procesar resultados tabla
    mae_base = ablation_maes["A (Base)"]
    ablation_results = []
    for conf_name, mae in ablation_maes.items():
        ganancia = mae_base - mae
        ablation_results.append({"Configuración": conf_name, "MAE (°C)": mae, "Ganancia (°C)": max(0, ganancia)})
        
    df_ablation = pd.DataFrame(ablation_results).set_index("Configuración")
    print_markdown_table(df_ablation, "TABLA 8: ESTUDIO DE ABLACIÓN - CONTRIBUCIÓN INCREMENTAL (2025)")

def main_baselines():
    print("=" * 75)
    print("  PHASE 2: ENSEMBLE BASELINES & ABLATION STUDY (ZERO-LEAKAGE)")
    print("=" * 75)
    
    df_raw = pd.read_csv(DATASET_PATH, parse_dates=["datetime"])
    df_lagged = generate_optimized_lags(df_raw)

    run_ablation_study(df_lagged)

    df_train = df_lagged[df_lagged["datetime"] < "2024-01-01"]
    y_train_hist = df_train["TT"].values
    train_naive_denominator = np.mean(np.abs(y_train_hist[24:] - y_train_hist[:-24]))

    features = [
        "hour_sin", "hour_cos", "month_sin", "month_cos", "wind_u", "wind_v", "dew_point_dep",
        "lag_TT_anomaly_1h", "lag_TT_anomaly_24h", "lag_TT_anomaly_72h", "lag_TT_anomaly_168h", "lag_TT_anomaly_360h"
    ]

    """
    ================================================================================
    [JUSTIFICACIÓN DE HIPERPARÁMETROS PARA EL ARTÍCULO]
    Los hiperparámetros declarados a continuación fueron obtenidos mediante un 
    proceso de Optimización Bayesiana utilizando el framework Optuna (TPE).
    - Espacio de Búsqueda: Miles de combinaciones evaluadas.
    - Partición: Conjunto de Validación estricto del año 2024 (Zero-leakage).
    - Función Objetivo: Minimización del Error Absoluto Medio (MAE).
    ================================================================================
    """
    models = {
        "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
        "XGBoost": XGBRegressor(n_estimators=124, learning_rate=0.062652, max_depth=5, min_child_weight=2,
                                reg_alpha=0.003322, reg_lambda=0.010598, random_state=42, n_jobs=-1),
        "LightGBM": LGBMRegressor(n_estimators=250, learning_rate=0.044129, max_depth=4, num_leaves=62,
                                  min_child_samples=22, reg_alpha=0.010630, reg_lambda=0.628150, verbosity=-1, n_jobs=-1, random_state=42),
        "CatBoost": CatBoostRegressor(iterations=201, learning_rate=0.070045, depth=4, l2_leaf_reg=2.477102, random_seed=42, verbose=0)
    }

    print(f"\n[INFO] Ajustando modelos base sobre {df_train.shape[0]} muestras...")
    for name, model in models.items():
        model.fit(df_train[features], df_train["TT_anomaly"])

    start_dates = pd.date_range(start="2025-01-01 00:00:00", end="2025-12-01 00:00:00", freq="MS")
    metrics_summary = {name: {"MAE": [], "RMSE": [], "MASE": [], "MBE": [], "R2": []} for name in list(models.keys()) + ["Seasonal Naive"]}
    
    evaluation_horizons = [1, 24, 72, 168, 336, 720]
    degradation_curves = {name: {h: [] for h in evaluation_horizons} for name in metrics_summary.keys()}
    june_start = pd.to_datetime("2025-06-01 00:00:00")
    june_visualization_data = {}

    print(f"[EVALUATION] Iniciando simulación multi-ventana deslizante (Año 2025)...")
    for start_date in start_dates:
        context_data = df_lagged[df_lagged["datetime"] < start_date]
        future_data = df_lagged[df_lagged["datetime"] >= start_date].head(HORIZON)
        y_real = future_data["TT"].values
        clim_val = future_data["TT_climatology"].values
        anomaly_history = context_data["TT_anomaly"].tail(360).tolist()

        for name, model in models.items():
            pred_anomalies = run_recursive_inference(model, anomaly_history, future_data, features, horizon=HORIZON)
            pred_real = pred_anomalies + clim_val

            metrics_summary[name]["MAE"].append(mean_absolute_error(y_real, pred_real))
            metrics_summary[name]["RMSE"].append(np.sqrt(mean_squared_error(y_real, pred_real)))
            metrics_summary[name]["MASE"].append(mean_absolute_error(y_real, pred_real) / train_naive_denominator)
            metrics_summary[name]["MBE"].append(np.mean(pred_real - y_real))
            metrics_summary[name]["R2"].append(r2_score(y_real, pred_real))
            
            for h in evaluation_horizons: 
                degradation_curves[name][h].append(mean_absolute_error(y_real[:h], pred_real[:h]))

            if start_date == june_start: june_visualization_data[name] = pred_real

        # Naive Benchmark
        naive_preds = np.tile(context_data["TT"].tail(24).values, 30)[:HORIZON]
        metrics_summary["Seasonal Naive"]["MAE"].append(mean_absolute_error(y_real, naive_preds))
        metrics_summary["Seasonal Naive"]["RMSE"].append(np.sqrt(mean_squared_error(y_real, naive_preds)))
        metrics_summary["Seasonal Naive"]["MASE"].append(mean_absolute_error(y_real, naive_preds) / train_naive_denominator)
        metrics_summary["Seasonal Naive"]["MBE"].append(np.mean(naive_preds - y_real))
        metrics_summary["Seasonal Naive"]["R2"].append(r2_score(y_real, naive_preds))
        
        for h in evaluation_horizons: 
            degradation_curves["Seasonal Naive"][h].append(mean_absolute_error(y_real[:h], naive_preds[:h]))

        if start_date == june_start:
            june_visualization_data["Seasonal Naive"] = naive_preds
            june_visualization_data["Observed"] = y_real

    # Tabla Final de Rendimiento
    final_results = []
    for name in metrics_summary.keys():
        final_results.append({
            "Model": name, "MAE": np.mean(metrics_summary[name]["MAE"]), 
            "RMSE": np.mean(metrics_summary[name]["RMSE"]),
            "MASE": np.mean(metrics_summary[name]["MASE"]), 
            "MBE": np.mean(metrics_summary[name]["MBE"]),
            "R2": np.mean(metrics_summary[name]["R2"])
        })

    results_df = pd.DataFrame(final_results).set_index("Model")
    print_markdown_table(results_df, "TABLA 7: RENDIMIENTO METROLÓGICO (BASELINES - PROMEDIO TEST 2025)")

    # Gráficos
    print("\n[VISUALIZATION] Exportando gráficos estandarizados...")
    plt.figure(figsize=(12, 5))
    plt.plot(june_visualization_data["Observed"], label="Observed Temperature", color="black", alpha=0.7, linewidth=1.2)
    plt.plot(june_visualization_data["XGBoost"], label="XGBoost Anomaly Forecast", color="#1f77b4", linewidth=1.3)
    plt.plot(june_visualization_data["LightGBM"], label="LightGBM Anomaly Forecast", color="#2ca02c", linewidth=1.3)
    plt.plot(june_visualization_data["Seasonal Naive"], label="Seasonal Naive Benchmark", color="gray", linestyle="--", alpha=0.6, linewidth=1.0)
    plt.title("Forecast Performance vs. Ground Truth (June 2025)", fontweight="bold", fontsize=10)
    plt.ylabel("Temperature (°C)"); plt.xlabel("Horizon (Hours)")
    plt.legend(loc="upper right", frameon=True)
    plt.tight_layout()
    plt.savefig("Fig_04_Comparacion_Pronosticos_Junio.png", dpi=300)
    plt.close()

    plt.figure(figsize=(8, 4.5))
    h_labels = ["1h", "24h", "3d (72h)", "7d (168h)", "14d (336h)", "30d (720h)"]
    colors = {"XGBoost": "#1f77b4", "Random Forest": "#e67e22", "LightGBM": "#2ca02c", "CatBoost": "#9467bd", "Seasonal Naive": "gray"}
    markers = {"XGBoost": "o-", "Random Forest": "s-", "LightGBM": "D-", "CatBoost": "v-", "Seasonal Naive": "x--"}

    for name in degradation_curves.keys():
        mean_errors = [np.mean(degradation_curves[name][h]) for h in evaluation_horizons]
        plt.plot(h_labels, mean_errors, markers[name], label=name, color=colors[name], alpha=0.7 if name == "Seasonal Naive" else 1.0, linewidth=1.5)

    plt.title("Average Error Degradation Analysis (Mean Cumulative MAE 2025)", fontweight="bold", fontsize=10)
    plt.ylabel("Mean Cumulative MAE (°C)"); plt.xlabel("Forecast Horizon")
    plt.legend(loc="lower right", frameon=True)
    plt.tight_layout()
    plt.savefig("Fig_05_Degradacion_Error_Acumulado.png", dpi=300)
    plt.close()

    print("[MEMORY] Liberando RAM Fase 2...")
    del models, df_lagged, df_train, df_raw
    gc.collect()

main_baselines()

In [ ]:
# ==============================================================================
# FASE 3: DEEP RECURRENT LEARNING (LSTM)
# ==============================================================================
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

EPOCHS = 35                 
LEARNING_RATE = 0.0005      
HIDDEN_SIZE = 64            
NUM_LAYERS = 2              
DROPOUT = 0.2               
BATCH_SIZE = 128            
SEQ_LENGTH = 168            

FEATURES_LSTM = ["TT_anomaly", "hour_sin", "hour_cos", "month_sin", "month_cos"]

class ClosedLoopTimeSeriesDataset(Dataset):
    def __init__(self, X: np.ndarray, seq_length: int):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.seq_length = seq_length
    def __len__(self) -> int: return len(self.X) - self.seq_length
    def __getitem__(self, i: int): return self.X[i:i+self.seq_length], self.X[i+self.seq_length, 0].unsqueeze(0)

class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

def run_recursive_lstm_inference(model, initial_scaled_history, future_exog_scaled, scaler, device, horizon=720):
    model.eval()
    predictions_scaled = []
    curr_hist = initial_scaled_history.copy()
    
    with torch.no_grad():
        for i in range(horizon):
            inp = torch.tensor(curr_hist, dtype=torch.float32).unsqueeze(0).to(device)
            pred_scaled = model(inp).cpu().numpy().flatten()[0]
            predictions_scaled.append(pred_scaled)
            
            next_step = np.zeros(curr_hist.shape[1])
            next_step[0] = pred_scaled  
            next_step[1:] = future_exog_scaled[i, 1:]  
            curr_hist = np.vstack((curr_hist[1:], next_step))
            
    predictions_scaled = np.array(predictions_scaled).reshape(-1, 1)
    dummy = np.zeros((horizon, curr_hist.shape[1]))
    dummy[:, 0] = predictions_scaled[:, 0]
    return scaler.inverse_transform(dummy)[:, 0]

def main_lstm():
    print("\n" + "=" * 75)
    print("  PHASE 3: DEEP RECURRENT LEARNING (LSTM CLOSED-LOOP)")
    print("=" * 75)
    
    df_raw = pd.read_csv(DATASET_PATH, parse_dates=["datetime"])
    df_train = df_raw[df_raw["datetime"] < "2024-01-01"]
    df_val = df_raw[(df_raw["datetime"] >= "2024-01-01") & (df_raw["datetime"] < "2025-01-01")]
    train_naive_denom = np.mean(np.abs(df_train["TT"].values[24:] - df_train["TT"].values[:-24]))

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(df_train[FEATURES_LSTM].values)
    val_scaled = scaler.transform(df_val[FEATURES_LSTM].values)

    train_loader = DataLoader(ClosedLoopTimeSeriesDataset(train_scaled, SEQ_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(ClosedLoopTimeSeriesDataset(val_scaled, SEQ_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[SYSTEM] LSTM Device: {device}")

    model = LSTMRegressor(len(FEATURES_LSTM), HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    train_losses, val_losses = [], []

    print("[TRAINING] Iniciando optimización LSTM...")
    for epoch in range(EPOCHS):
        model.train()
        epoch_train_loss = 0.0
        for x_b, y_b in train_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_b), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_train_loss += loss.item()

        model.eval()
        epoch_val_loss = 0.0
        with torch.no_grad():
            for x_v, y_v in val_loader:
                x_v, y_v = x_v.to(device), y_v.to(device)
                epoch_val_loss += criterion(model(x_v), y_v).item()

        avg_train, avg_val = epoch_train_loss / len(train_loader), epoch_val_loss / len(val_loader)
        train_losses.append(avg_train); val_losses.append(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), "best_lstm_model.pth")

    model.load_state_dict(torch.load("best_lstm_model.pth"))

    start_dates = pd.date_range(start="2025-01-01 00:00:00", end="2025-12-01 00:00:00", freq="MS")
    metrics_summary = {"MAE": [], "RMSE": [], "MASE": [], "MBE": [], "R2": []}
    
    june_start = pd.to_datetime("2025-06-01 00:00:00")
    june_data = {}

    print("[EVALUATION] Evaluando ventanas LSTM en 2025...")
    for start_date in start_dates:
        context_data = df_raw[df_raw["datetime"] < start_date]
        future_data = df_raw[df_raw["datetime"] >= start_date].head(HORIZON)

        y_real = future_data["TT"].values
        context_scaled = scaler.transform(context_data[FEATURES_LSTM].values)
        future_scaled = scaler.transform(future_data[FEATURES_LSTM].values)

        pred_anomalies = run_recursive_lstm_inference(model, context_scaled[-SEQ_LENGTH:], future_scaled, scaler, device, horizon=HORIZON)
        pred_real = pred_anomalies + future_data["TT_climatology"].values

        metrics_summary["MAE"].append(mean_absolute_error(y_real, pred_real))
        metrics_summary["RMSE"].append(np.sqrt(mean_squared_error(y_real, pred_real)))
        metrics_summary["MASE"].append(mean_absolute_error(y_real, pred_real) / train_naive_denom)
        metrics_summary["MBE"].append(np.mean(pred_real - y_real))
        metrics_summary["R2"].append(r2_score(y_real, pred_real))
        
        if start_date == june_start:
            june_data["LSTM"], june_data["Observed"] = pred_real, y_real

    results_df = pd.DataFrame([{
        "Model": "LSTM (Anomaly)", "MAE": np.mean(metrics_summary["MAE"]), 
        "RMSE": np.mean(metrics_summary["RMSE"]), "MASE": np.mean(metrics_summary["MASE"]), 
        "MBE": np.mean(metrics_summary["MBE"]), "R2": np.mean(metrics_summary["R2"])
    }]).set_index("Model")
    print_markdown_table(results_df, "RENDIMIENTO LSTM (2025)")

    print("\n[VISUALIZATION] Exportando gráficos LSTM...")
    plt.figure(figsize=(12, 5))
    plt.plot(june_data["Observed"], label="Observed Temperature", color="black", alpha=0.7, linewidth=1.2)
    plt.plot(june_data["LSTM"], label="LSTM Anomaly Forecast", color="#8e44ad", linewidth=1.5)
    plt.title("LSTM Performance vs. Ground Truth (June 2025)", fontweight="bold")
    plt.ylabel("Temperature (°C)"); plt.xlabel("Horizon (Hours)")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.savefig("Fig_06_LSTM_Bucle_Cerrado.png", dpi=300)
    plt.close()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    ax1.plot(train_losses, label="Training Loss (MSE)", color="#2980b9", linewidth=1.5)
    ax1.plot(val_losses, label="Validation Loss (MSE)", color="#e67e22", linestyle="--", linewidth=1.5)
    ax1.set_title("Neural Network Convergence Curve", fontweight="bold")
    ax1.set_ylabel("Loss"); ax1.legend(loc="upper right")

    residuals = june_data["Observed"] - june_data["LSTM"]
    sns.histplot(residuals, kde=True, color="#16a085", edgecolor="black", alpha=0.7, ax=ax2)
    ax2.axvline(0, color="red", linestyle=":", linewidth=1.5, label="Zero Error Reference")
    ax2.set_title("Distribution of Forecast Residuals (June 2025)", fontweight="bold")
    ax2.set_xlabel("Residual Error (°C)"); ax2.set_ylabel("Frequency"); ax2.legend()
    plt.tight_layout()
    plt.savefig("Fig_07_LSTM_Optimizacion_Residuos.png", dpi=300)
    plt.close()

    # VRAM Cleanup Strict
    del model, train_loader, val_loader, scaler, df_raw, df_train, df_val
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()

main_lstm()

In [ ]:
# ==============================================================================
# FASE 4: TEMPORAL FUSION TRANSFORMER (TFT PROBABILÍSTICO)
# ==============================================================================
import lightning.pytorch as pl
from matplotlib.gridspec import GridSpec
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

pl.seed_everything(42, workers=True)
torch.set_float32_matmul_precision("medium")

TFT_EPOCHS = 15                 
TFT_LR = 0.000011      
TFT_HIDDEN = 16            
TFT_HEADS = 2     
TFT_DROPOUT = 0.180094          
TFT_BATCH = 64             
SEQ_PAST = 360       

def calc_q_risk(y_true, y_quant, q):
    loss = np.maximum(q * (y_true - y_quant), (q - 1.0) * (y_true - y_quant))
    return float(2.0 * np.sum(loss) / np.sum(np.abs(y_true)))

def main_tft():
    print("\n" + "=" * 75)
    print("  PHASE 4: TEMPORAL FUSION TRANSFORMER (PROBABILISTIC DEEP LEARNING)")
    print("=" * 75)
    
    df_raw = pd.read_csv(DATASET_PATH, parse_dates=["datetime"])
    df_raw["hour_str"] = df_raw["datetime"].dt.hour.astype(str)
    df_raw["month_str"] = df_raw["datetime"].dt.month.astype(str)
    df_raw["day_of_year_sin"] = np.sin(2 * np.pi * df_raw["datetime"].dt.dayofyear / 365.25)
    df_raw["day_of_year_cos"] = np.cos(2 * np.pi * df_raw["datetime"].dt.dayofyear / 365.25)

    df_train = df_raw[df_raw["datetime"] < "2024-01-01"].copy()
    df_val = df_raw[(df_raw["datetime"] >= "2024-01-01") & (df_raw["datetime"] < "2025-01-01")].copy()
    df_test_full = df_raw[df_raw["datetime"] >= "2024-12-15"].copy() 

    train_naive_denom = np.mean(np.abs(df_train["TT"].values[24:] - df_train["TT"].values[:-24]))

    scaler = StandardScaler()
    cols_to_scale = ["HR", "PP", "RR", "wind_u", "wind_v", "dew_point_dep"]
    scaler.fit(df_train[cols_to_scale])
    df_train[cols_to_scale] = scaler.transform(df_train[cols_to_scale])
    df_val[cols_to_scale] = scaler.transform(df_val[cols_to_scale])
    df_test_full[cols_to_scale] = scaler.transform(df_test_full[cols_to_scale])

    training_dataset = TimeSeriesDataSet(
        df_train, time_idx="time_idx", target="TT_anomaly", group_ids=["group_id"],
        min_encoder_length=SEQ_PAST, max_encoder_length=SEQ_PAST,
        min_prediction_length=HORIZON, max_prediction_length=HORIZON,
        static_categoricals=["group_id"], time_varying_known_categoricals=["hour_str", "month_str"],
        time_varying_known_reals=["hour_sin", "hour_cos", "day_of_year_sin", "day_of_year_cos", "month_sin", "month_cos", "TT_climatology"],
        time_varying_unknown_reals=["TT_anomaly", "HR", "PP", "RR", "wind_u", "wind_v", "dew_point_dep"],
        target_normalizer=None, add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
    )

    val_dataset = TimeSeriesDataSet.from_dataset(training_dataset, df_val, predict=False, stop_randomization=True)
    num_workers = 2 if torch.cuda.is_available() else 0
    train_dataloader = training_dataset.to_dataloader(train=True, batch_size=TFT_BATCH, num_workers=num_workers)
    val_dataloader = val_dataset.to_dataloader(train=False, batch_size=TFT_BATCH * 4, num_workers=num_workers)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[SYSTEM] TFT Device: {device}")

    tft = TemporalFusionTransformer.from_dataset(
        training_dataset, learning_rate=TFT_LR, hidden_size=TFT_HIDDEN,
        attention_head_size=TFT_HEADS, dropout=TFT_DROPOUT, hidden_continuous_size=TFT_HIDDEN // 2,
        output_size=7, loss=QuantileLoss(), reduce_on_plateau_patience=2,
    )

    logger = CSVLogger("logs", name="tft_logs")
    checkpoint = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1, dirpath="./")
    early_stop = EarlyStopping(monitor="val_loss", patience=2, mode="min")

    trainer = pl.Trainer(
        max_epochs=TFT_EPOCHS, accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1, precision="32", gradient_clip_val=0.1,
        callbacks=[early_stop, checkpoint], logger=logger, enable_progress_bar=True
    )

    print("[TRAINING] Optimizando pesos probabilísticos del TFT...")
    trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

    best_tft = TemporalFusionTransformer.load_from_checkpoint(checkpoint.best_model_path).to(device)

    start_dates = pd.date_range(start="2025-01-01 00:00:00", end="2025-12-01 00:00:00", freq="MS")
    metrics_tft = {"MAE": [], "RMSE": [], "MASE": [], "MBE": [], "R2": [], "PICP": [], "MPIW": []}
    june_start, june_data, june_raw = pd.to_datetime("2025-06-01 00:00:00"), {}, None

    print("[EVALUATION] Ejecutando inferencia probabilística 2025...")
    for start_date in start_dates:
        window_df = df_test_full[(df_test_full["datetime"] >= start_date - pd.Timedelta(hours=SEQ_PAST)) & 
                                 (df_test_full["datetime"] < start_date + pd.Timedelta(hours=HORIZON))].copy()

        if len(window_df) < (SEQ_PAST + HORIZON): continue

        w_dataset = TimeSeriesDataSet.from_dataset(training_dataset, window_df, predict=True, stop_randomization=True)
        w_dataloader = w_dataset.to_dataloader(train=False, batch_size=1, num_workers=0)

        with torch.no_grad():
            raw_preds = best_tft.predict(w_dataloader, mode="raw", return_x=True)

        q10 = raw_preds.output.prediction[0, :, 1].cpu().numpy()
        q50 = raw_preds.output.prediction[0, :, 3].cpu().numpy()
        q90 = raw_preds.output.prediction[0, :, 5].cpu().numpy()

        future_data = window_df.tail(HORIZON)
        y_real, clim_future = future_data["TT"].values, future_data["TT_climatology"].values
        p10, p50, p90 = q10 + clim_future, q50 + clim_future, q90 + clim_future

        metrics_tft["MAE"].append(mean_absolute_error(y_real, p50))
        metrics_tft["RMSE"].append(np.sqrt(mean_squared_error(y_real, p50)))
        metrics_tft["MASE"].append(mean_absolute_error(y_real, p50) / train_naive_denom)
        metrics_tft["MBE"].append(np.mean(p50 - y_real))
        metrics_tft["R2"].append(r2_score(y_real, p50))
        
        covered = (y_real >= p10) & (y_real <= p90)
        metrics_tft["PICP"].append(float(np.mean(covered)))
        metrics_tft["MPIW"].append(float(np.mean(p90 - p10)))

        if start_date == june_start:
            june_data["Observed"] = y_real
            june_data["TFT_P10"], june_data["TFT_P50"], june_data["TFT_P90"] = p10, p50, p90
            june_raw = raw_preds

    results_df = pd.DataFrame([{
        "Model": "Temporal Fusion Transformer", "MAE": np.mean(metrics_tft["MAE"]), 
        "RMSE": np.mean(metrics_tft["RMSE"]), "MASE": np.mean(metrics_tft["MASE"]), 
        "MBE": np.mean(metrics_tft["MBE"]), "R2": np.mean(metrics_tft["R2"]),
        "PICP (Cov %)": np.mean(metrics_tft["PICP"]), "MPIW (Width)": np.mean(metrics_tft["MPIW"])
    }]).set_index("Model")
    print_markdown_table(results_df, "RENDIMIENTO PROBABILÍSTICO TFT (2025)")

    print("\n[VISUALIZATION] Exportando panel gráfico SOTA...")
    fig = plt.figure(figsize=(18, 12))
    gs = GridSpec(2, 3, figure=fig)

    ax_a = fig.add_subplot(gs[0, 0:2])
    ax_a.plot(june_data["Observed"], label="Observed Temperature", color="black", lw=1.2, alpha=0.8)
    ax_a.plot(june_data["TFT_P50"], label="TFT Predict (p50 Median)", color="#d32f2f", lw=1.5)
    ax_a.fill_between(range(HORIZON), june_data["TFT_P10"], june_data["TFT_P90"], color="#d32f2f", alpha=0.15, label="80% Confidence Interval")
    ax_a.set_title("A) Probabilistic Subseasonal Forecast (June 2025)", fontweight="bold")
    ax_a.set_ylabel("Temperature (°C)"); ax_a.legend(loc="upper right", frameon=True)

    ax_b = fig.add_subplot(gs[0, 2])
    try:
        metrics_csv = pd.read_csv(f"{logger.log_dir}/metrics.csv")
        val_loss, train_loss = metrics_csv["val_loss"].dropna().values, metrics_csv["train_loss_step"].dropna().values
        ax_b.plot(np.linspace(1, len(val_loss), len(train_loss)), train_loss, label="Train Loss", alpha=0.3, color="#34495e")
        ax_b.plot(range(1, len(val_loss) + 1), val_loss, label="Validation Loss", color="red", lw=1.5)
        ax_b.set_title("B) Anomaly Convergence Curve", fontweight="bold")
        ax_b.legend(loc="upper right")
    except: pass

    ax_c = fig.add_subplot(gs[1, 0])
    sns.histplot(june_data["Observed"] - june_data["TFT_P50"], kde=True, color="#2980b9", ax=ax_c, edgecolor="black")
    ax_c.axvline(0, color="red", linestyle=":", lw=1.5, label="Zero Error Reference")
    ax_c.set_title("C) Reconstructed Residual Errors (June 2025)", fontweight="bold")

    try:
        interp = best_tft.interpret_output(june_raw.output, reduction="sum")
        enc_vals, dec_vals = interp["encoder_variables"].cpu().numpy(), interp["decoder_variables"].cpu().numpy()
        enc_vals, dec_vals = enc_vals / enc_vals.sum() * 100, dec_vals / dec_vals.sum() * 100

        ax_d1 = fig.add_subplot(gs[1, 1])
        enc_idx = np.argsort(enc_vals)
        ax_d1.barh(range(len(enc_idx)), enc_vals[enc_idx], color="#1abc9c", edgecolor="black")
        ax_d1.set_yticks(range(len(enc_idx)))
        ax_d1.set_yticklabels([best_tft.encoder_variables[i] for i in enc_idx], fontsize=9)
        ax_d1.set_title("D1) Anomaly Encoder Importance", fontweight="bold")

        ax_d2 = fig.add_subplot(gs[1, 2])
        dec_idx = np.argsort(dec_vals)
        ax_d2.barh(range(len(dec_idx)), dec_vals[dec_idx], color="#f39c12", edgecolor="black")
        ax_d2.set_yticks(range(len(dec_idx)))
        ax_d2.set_yticklabels([best_tft.decoder_variables[i] for i in dec_idx], fontsize=9)
        ax_d2.set_title("D2) Anomaly Decoder Importance", fontweight="bold")
    except: pass

    plt.tight_layout()
    plt.savefig("Fig_08_TFT_Dashboard_Atencion.png", dpi=300)
    plt.close()

    print("\n✅ PROCESO FINALIZADO CON ÉXITO ABSOLUTO.")

main_tft()